# McKesson Decision Knowledge Graph — Interactive Exploration

This notebook walks through the same three demos but lets you tweak inputs and re-run cells.

**Setup:** make sure your venv is activated and dependencies are installed (`pip install -r requirements.txt`).

In [1]:
import sys, os, json
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT))

from dkg.graph_builder import build_graph, summary
from dkg.queries import (
    list_decisions_by_team, get_decision_context, blast_radius,
    input_criticality, cross_team_chains, kpi_to_decisions, exception_impact
)
from dkg.decision_engine import DecisionEngine
from dkg.visualize import render_interactive

g = build_graph(PROJECT_ROOT / 'data' / 'us_pharma_dkg.json')
summary(g)

{'node:team': 12,
 'node:decision': 12,
 'node:data_input': 14,
 'node:constraint': 8,
 'node:artifact': 12,
 'node:exception': 7,
 'node:kpi': 9,
 'node:system': 6,
 'edge:owns': 12,
 'edge:consumes': 21,
 'edge:bounded_by': 10,
 'edge:produces': 12,
 'edge:optimizes_for': 13,
 'edge:executed_in': 7,
 'edge:feeds_into': 9,
 'edge:overridden_by': 10,
 'total_nodes': 80,
 'total_edges': 94}

## 1. Visualize

Render the interactive HTML in this folder, then open it in your browser.

In [2]:
out = render_interactive(g, PROJECT_ROOT / 'outputs' / 'dkg_interactive.html')
print(f'Open {out} in a browser')

Open /Users/akshatkharaya/Documents/Coding/CompanyDKG/outputs/dkg_interactive.html in a browser


## 2. Query the graph

Try changing the team_id or decision_id to explore different parts of the BU.

In [3]:
# What does a particular team own?
list_decisions_by_team(g, 'team_treasury')

[{'id': 'dec_cash_coverage',
  'name': 'Cash coverage & funding decision',
  'cadence': 'daily'}]

In [4]:
# Blast radius — if X breaks, what else does?
blast_radius(g, 'dec_demand_forecast', max_hops=3)

[{'id': 'dec_place_po',
  'name': 'Place weekly POs with manufacturers',
  'hops_away': 2,
  'type': 'decision'},
 {'id': 'dec_warehouse_schedule',
  'name': 'Daily warehouse labor & throughput plan',
  'hops_away': 2,
  'type': 'decision'}]

In [5]:
# Most critical inputs across the whole BU
import pandas as pd

pd.DataFrame(input_criticality(g)).head(10)
pd.DataFrame(input_criticality(g)).head(10)

,id,name,source_system,critical_dependencies,supplementary_dependencies,total,decisions
0,inp_order_volume,Daily order volume,ERP,2,1,3,"[Weekly SKU-level demand forecast, Daily wareh..."
1,inp_customer_pay_history,Customer payment patterns,ERP,2,1,3,"[Customer credit limit approval, AR collection..."
2,inp_ar_aging,AR aging buckets,ERP,2,0,2,"[AR collections action, Daily cash flow forecast]"
3,inp_market_price,Manufacturer list & contract prices,Contract Mgmt,1,1,2,"[Place weekly POs with manufacturers, Customer..."
4,inp_historical_sales,Historical sales transactions,ERP,1,1,2,"[Weekly SKU-level demand forecast, Daily cash ..."
5,inp_demand_forecast,SKU demand forecast,Demand Planning Tool,1,0,1,[Place weekly POs with manufacturers]
6,inp_current_inventory,Current on-hand inventory,WMS,1,0,1,[Place weekly POs with manufacturers]
7,inp_open_pos,Open POs in transit,ERP,1,0,1,[Place weekly POs with manufacturers]
8,inp_traffic_weather,Traffic & weather feeds,External APIs,1,0,1,[Daily delivery route optimization]
9,inp_competitor_price,Competitor price intelligence,Market Intel,1,0,1,[Customer pricing changes]


In [6]:
# Cross-team dependency chains
for c in cross_team_chains(g):
    print(f"{c['from_team']} -> {c['to_team']}  via  {c['via_artifact']}")

Inventory Purchasing -> FP&A  via  Weekly PO batch
Demand Planning -> Inventory Purchasing  via  13-week demand forecast
Demand Planning -> Warehouse Operations  via  13-week demand forecast
Pricing & Contracts -> FP&A  via  Monthly price update
Supplier Management -> Inventory Purchasing  via  Updated supplier contract
Credit Risk -> AR Collections  via  Customer credit decision
AR Collections -> FP&A  via  Weekly collections action list
FP&A -> Treasury  via  Daily cash flow forecast
FP&A -> Capital Planning  via  Daily cash flow forecast


## 3. Autonomous decisions

Edit the world state, re-run, and watch behavior change.

In [7]:
with open(PROJECT_ROOT / 'data' / 'world_state.json') as f:
    world = json.load(f)
world['active_exceptions'] = set(world['active_exceptions'])
engine = DecisionEngine(g, world)

rec = engine.run('dec_place_po', sku='SKU_002_lipitor_20')
print(rec.pretty())


DECISION: Place weekly POs with manufacturers  (dec_place_po)
Recommended action : Place PO for SKU=SKU_002_lipitor_20 qty=5360
Confidence         : 75%

Rationale:
  - Base target = forecast(8000) - on_hand(800) - in_transit(2000) + safety_min(1500) = 6700
  - DSI reduction directive active: cut target 6700 -> 5360

Inputs consulted:
  - SKU demand forecast
  - Current on-hand inventory
  - Open POs in transit
  - Manufacturer list & contract prices

Active exceptions:
  - CFO DSI reduction directive

Expected KPI impact:
  - DSI: decrease
  - Fill rate: maintain


In [8]:
# What if we add a stockout-risk exception on top of the DSI directive?
world2 = {**world, 'active_exceptions': world['active_exceptions'] | {'exc_stockout_risk'}}
engine2 = DecisionEngine(g, world2)
print(engine2.run('dec_place_po', sku='SKU_002_lipitor_20').pretty())


DECISION: Place weekly POs with manufacturers  (dec_place_po)
Recommended action : Place PO for SKU=SKU_002_lipitor_20 qty=6968
Confidence         : 75%
Escalate to        : Inventory Purchasing

Rationale:
  - Base target = forecast(8000) - on_hand(800) - in_transit(2000) + safety_min(1500) = 6700
  - DSI reduction directive active: cut target 6700 -> 5360
  - Stockout risk active: increase target 5360 -> 6968

Inputs consulted:
  - SKU demand forecast
  - Current on-hand inventory
  - Open POs in transit
  - Manufacturer list & contract prices

Active exceptions:
  - CFO DSI reduction directive
  - Imminent stockout alert

Expected KPI impact:
  - DSI: decrease
  - Fill rate: maintain


In [9]:
# Run the full battery
for did in engine.supported_decisions():
    print(engine.run(did).pretty())


DECISION: Place weekly POs with manufacturers  (dec_place_po)
Recommended action : Place PO for SKU=SKU_001_amox_500 qty=8000
Confidence         : 75%

Rationale:
  - Base target = forecast(12000) - on_hand(3000) - in_transit(1500) + safety_min(2500) = 10000
  - DSI reduction directive active: cut target 10000 -> 8000

Inputs consulted:
  - SKU demand forecast
  - Current on-hand inventory
  - Open POs in transit
  - Manufacturer list & contract prices

Active exceptions:
  - CFO DSI reduction directive

Expected KPI impact:
  - DSI: decrease
  - Fill rate: maintain

DECISION: Customer credit limit approval  (dec_credit_limit)
Recommended action : Approve credit limit $250,000 for cust_pharmacy_alpha
Confidence         : 80%

Rationale:
  - Customer tier=tier_2 -> default limit $200,000
  - Credit score = 720, avg days-late = 3
  - Strong credit profile; approve up to $250,000

Inputs consulted:
  - Credit bureau report
  - Customer payment patterns

Expected KPI impact:
  - Bad debt 